In [1]:
!curl -L -o wine-quality-dataset.zip https://www.kaggle.com/api/v1/datasets/download/yasserh/wine-quality-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 21984  100 21984    0     0  38560      0 --:--:-- --:--:-- --:--:-- 38560


In [2]:
!unzip /content/wine-quality-dataset.zip

Archive:  /content/wine-quality-dataset.zip
  inflating: WineQT.csv              


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("WineQT.csv")

In [5]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [6]:
df = df.drop("Id",axis=1)

In [7]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [8]:
df.isna().sum()

,0
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


In [9]:
X = df.drop('quality',axis=1).values
y = df['quality'].copy().values

In [13]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
import sklearn
from sklearn.model_selection import train_test_split

In [14]:
if torch.cuda.is_available():
  device = 'cuda'
if torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [15]:
X_train_full,X_test,y_train_full,y_test = train_test_split(X,y,test_size=0.2)

X_train = torch.FloatTensor(X_train_full)
X_test = torch.FloatTensor(X_test)

y_train = torch.FloatTensor(y_train_full).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [16]:
means = X_train.mean(dim=0,keepdims=True)
stds = X_train.std(dim=0,keepdims=True)

X_train = (X_train - means) / stds
X_test = (X_test - means) / stds

In [17]:
train_dataset = TensorDataset(X_train,y_train)
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

In [18]:
n_features = X_train.shape[1]

model = nn.Sequential(
    nn.Linear(n_features,64),
    nn.ReLU(),
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,1)
)

In [19]:
model.to(device)

Sequential(
  (0): Linear(in_features=11, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)

In [21]:
learning_rate = 0.02
n_epochs = 50

optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)
mse = nn.MSELoss()

def train(model,optimizer,criterion,train_loader,n_epochs):
  model.train()
  for epoch in range(n_epochs):
    total_loss = 0
    for X_batch,y_batch in train_loader:
      X_batch,y_batch = X_batch.to(device),y_batch.to(device)

      y_pred = model(X_batch)
      loss = criterion(y_pred,y_batch)
      total_loss += loss.item()

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()


    mean_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{n_epochs}, Loss:{mean_loss:.4f}")

In [22]:
train(model,optimizer,mse,train_loader,n_epochs)

Epoch 1/50, Loss:5.0587
Epoch 2/50, Loss:0.7448
Epoch 3/50, Loss:0.5772
Epoch 4/50, Loss:0.5148
Epoch 5/50, Loss:0.4713
Epoch 6/50, Loss:0.4502
Epoch 7/50, Loss:0.4226
Epoch 8/50, Loss:0.4254
Epoch 9/50, Loss:0.4052
Epoch 10/50, Loss:0.3951
Epoch 11/50, Loss:0.4123
Epoch 12/50, Loss:0.3943
Epoch 13/50, Loss:0.4025
Epoch 14/50, Loss:0.4005
Epoch 15/50, Loss:0.4021
Epoch 16/50, Loss:0.3777
Epoch 17/50, Loss:0.3849
Epoch 18/50, Loss:0.3857
Epoch 19/50, Loss:0.3770
Epoch 20/50, Loss:0.3750
Epoch 21/50, Loss:0.3697
Epoch 22/50, Loss:0.3724
Epoch 23/50, Loss:0.3682
Epoch 24/50, Loss:0.3803
Epoch 25/50, Loss:0.3644
Epoch 26/50, Loss:0.3803
Epoch 27/50, Loss:0.3548
Epoch 28/50, Loss:0.3723
Epoch 29/50, Loss:0.3645
Epoch 30/50, Loss:0.3544
Epoch 31/50, Loss:0.3522
Epoch 32/50, Loss:0.3563
Epoch 33/50, Loss:0.3592
Epoch 34/50, Loss:0.3966
Epoch 35/50, Loss:0.3418
Epoch 36/50, Loss:0.3474
Epoch 37/50, Loss:0.3586
Epoch 38/50, Loss:0.3513
Epoch 39/50, Loss:0.3578
Epoch 40/50, Loss:0.3383
Epoch 41/

In [23]:
X_new = X_test[:5].to(device)

with torch.no_grad():
  y_pred = model(X_new)

print("Modelin texmini: ")
print(y_pred)

print("Real qiymeti: ")
print(y_test[:5])

Modelin texmini: 
tensor([[5.6836],
        [6.5573],
        [5.0373],
        [5.0474],
        [5.6219]])
Real qiymeti: 
tensor([[6.],
        [7.],
        [5.],
        [5.],
        [5.]])
